# Data Overview

### 1. DataFrame creation

In [1]:
# import libraries
from pyspark.sql import SparkSession

In [2]:
# Define bronze file location path

forenames = "../data/raw/forenames.csv/"
countries = "../data/raw/country_codes.csv"

In [3]:
# Create session

def create_spark_session(app_name:str):
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .getOrCreate()
    return spark

In [4]:
# Create a spark session to bronze layer
spark = create_spark_session("bronze_pipeline")

In [5]:
# Import libraries to set the schema
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import IntegerType, StringType, FloatType
from pyspark.sql.types import Row

In [6]:
# Structure the schema
nameTableSchema = StructType([
    StructField("forename", StringType(), False),
    StructField("gender", StringType(), False),
    StructField("country", StringType(), False),
    StructField("count", IntegerType(), False)
])

In [7]:
countryTableSchema = StructType([
    StructField("country_code", StringType(), False),
    StructField("country_name", StringType(), False)
])

In [8]:
# function to read files on spark

def read_files(path:str, schema:str):
    print("Starting file reading with Spark...")
    df = spark.read \
    .schema(schema) \
    .option("header", "true") \
    .csv(path)
    return df

In [9]:
# Create dataframe with Forenames data
df_fn = read_files(forenames, nameTableSchema)

Starting file reading with Spark...


In [10]:
# Create dataframe with Country code data
df_cc = read_files(countries, countryTableSchema)

Starting file reading with Spark...


In [11]:
df_fn.show(5)

+--------+------+-------+------+
|forename|gender|country| count|
+--------+------+-------+------+
|      Md|     M|     AE|132181|
|Muhammad|     M|     AE| 84884|
|Mohammed|     M|     AE| 51575|
|   Abdul|     M|     AE| 48408|
|     Ali|     M|     AE| 40866|
+--------+------+-------+------+
only showing top 5 rows


In [12]:
df_cc.show(5)

+------------+--------------------+
|country_code|        country_name|
+------------+--------------------+
|          AE|United Arab Emirates|
|          AF|         Afghanistan|
|          AL|             Albania|
|          AO|              Angola|
|          AR|           Argentina|
+------------+--------------------+
only showing top 5 rows


In [13]:
df_fn.printSchema()

root
 |-- forename: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- count: integer (nullable = true)



In [14]:
df_cc.printSchema()

root
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)



### 2. Record number

In [15]:
df_fn.count()

12470920

In [16]:
df_cc.count()

105

### 3. There are any null value

In [17]:
from pyspark.sql.functions import col, sum

df_null = df_fn.select([sum(col(c).isNull().cast('int')).alias(c) for c in df_fn.columns])
df_null.show()

+--------+-------+-------+-----+
|forename| gender|country|count|
+--------+-------+-------+-----+
|     291|2419135|  27572|    0|
+--------+-------+-------+-----+



### 4. Save Dataframe as parquet

In [18]:
df_fn.write.mode("overwrite").parquet("../data/bronze/df_forenames/")
df_cc.write.mode("overwrite").parquet("../data/bronze/df_countryCode/")